# Multi-Stage RAG Evaluation

Benchmarks every enabled retriever (per `config.yaml` → `retrievers.use`) plus the RRF fusion layer and the Cross-Encoder reranker, against queries generated from the patient medication registry.

**Prereqs:** all entries in `requirements.txt` installed. Heavy first-run cost: ColBERT (~1.6 GB), MedCPT × 2 (~880 MB), SPECTER 2 (~440 MB), Contriever (~440 MB), BioLinkBERT (~440 MB).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from src.utils import load_config
from main_pipeline import _selected_retrievers
from evaluate_metrics import (
    generate_eval_queries,
    evaluate,
    build_metrics_table,
    DEFAULT_K_VALUES,
)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
pd.set_option("display.width", 220)

## 1. Config & query sampling

Tune `N_QUERIES` and `CANDIDATE_LIMIT` to balance fidelity vs. runtime. With all 5 retrievers loaded, expect roughly 1–3 s per query on CPU and 0.3–1 s on GPU (after the first query, which pays the model-load cost).

In [ ]:
N_QUERIES = 50          # increase for tighter confidence on the means
CANDIDATE_LIMIT = 300   # candidate pool per query (pre-filtered by patient_id)
K_VALUES = (1, 3, 5, 10)

cfg = load_config()
print("Enabled retrievers:", _selected_retrievers(cfg))
print("Pipeline params  :", cfg["params"])

queries = generate_eval_queries(
    cfg["data"]["registry_csv"], n=N_QUERIES, seed=42
)
display(queries.head(5))
print(f"Total queries: {len(queries)}")

## 2. Run the full multi-stage pipeline on every query

Each query goes through:
1. **Stage 1** — every enabled retriever (latency tracked per backend)
2. **Stage 2** — Reciprocal Rank Fusion of the per-retriever outputs
3. **Stage 3** — Cross-Encoder rerank of the fused top-20

First-query latency includes one-time model loads and is dramatically larger than steady-state — read the **median** and **p90/p99** numbers rather than the first query.

In [ ]:
latencies, results_per_system, ground_truths = evaluate(
    queries, cfg, candidate_limit=CANDIDATE_LIMIT
)

## 3. Comparison table — every system in one frame

Rows are systems (one per enabled retriever, plus `RRF` after fusion and `FINAL` after Cross-Encoder rerank). Latency columns: per-retriever rows show the retriever's own wall time, `RRF`/`CrossEncoder` rows are just those stages, `FINAL` is the **full** per-query pipeline time.

In [ ]:
table = build_metrics_table(results_per_system, ground_truths, latencies, K_VALUES)
display(table)

out_path = "evaluation_results.csv"
table.to_csv(out_path)
print(f"\nSaved: {out_path}")

## 4. One detail table per system

Same data, sliced per-system so you can scan a single retriever's IR + latency profile in isolation.

In [ ]:
def _detail_view(table: pd.DataFrame, system: str, k_values=K_VALUES) -> pd.DataFrame:
    """Reshape a single row of the wide table into (metric × K) layout."""
    row = table.loc[system]
    ir_metrics = ["Hit", "P", "R", "NDCG"]
    detail = pd.DataFrame(
        {f"@{k}": [row[f"{m}@{k}"] for m in ir_metrics] for k in k_values},
        index=ir_metrics,
    )
    summary = pd.Series(
        {"MRR": row["MRR"], "avg_ms": row["avg_ms"], "p90_ms": row["p90_ms"], "p99_ms": row["p99_ms"]}
    )
    return detail, summary

for system in table.index:
    display(Markdown(f"### {system}"))
    detail, summary = _detail_view(table, system)
    display(detail)
    display(summary.to_frame("value").T)

## 5. Latency distribution

Boxplots of per-query latency per stage. Useful for spotting tail-latency culprits (e.g., ColBERT's PLAID build amortizes but the first call is slow).

In [ ]:
import matplotlib.pyplot as plt

stage_names = list(latencies.keys())
stage_data = [latencies[s] for s in stage_names]

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(stage_data, labels=stage_names, showfliers=False)
ax.set_ylabel("latency (ms)")
ax.set_title(f"Per-stage latency (N={len(queries)} queries, outliers hidden)")
ax.grid(True, alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Reading the results

- **Hit@K** — fraction of queries whose ground-truth note appears in the top K.
- **P@K / R@K** — with one ground-truth doc per query, `R@K == Hit@K` and `P@K = Hit@K / K`. The columns are kept separate so a future multi-truth setup drops in without refactoring.
- **MRR** — mean reciprocal rank of the first hit; higher means correct note ranks closer to the top.
- **NDCG@K** — binary-relevance NDCG; identical to MRR-style position weighting for single-truth queries.
- **`FINAL`** row reflects the cross-encoder-reranked top 5 over the RRF fused set. If `FINAL` is worse than `RRF`, the CE model is mis-scoring on your domain (worth trying a domain-tuned CE).
- **`RRF`** row over the **same K** as individual retrievers tells you how much fusion helps without the CE.